# 01 — HotpotQA Data Exploration

Load processed JSONL files and explore basic statistics.

In [ ]:
# Mount Drive (optional)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

EXP_PATH = Path("../data/processed/hotpotqa_experiments.jsonl")
CAL_PATH = Path("../data/processed/hotpotqa_calibration.jsonl")

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

experiments = load_jsonl(EXP_PATH)
calibration = load_jsonl(CAL_PATH)

print(f"Experiment samples: {len(experiments)}")
print(f"Calibration samples: {len(calibration)}")

In [ ]:
def avg_tokens(texts):
    return sum(len(t.split()) for t in texts) / len(texts)

questions = [s["question"] for s in experiments]
answers = [s["answer"] for s in experiments]

def context_tokens(sample):
    all_text = " ".join(
        " ".join(sents) for _, sents in sample["context"]
    )
    return len(all_text.split())

ctx_lengths = [context_tokens(s) for s in experiments]
sf_counts = [len(s["supporting_facts"]) for s in experiments]

print(f"Total samples              : {len(experiments)}")
print(f"Avg question length (words): {avg_tokens(questions):.1f}")
print(f"Avg answer length (words)  : {avg_tokens(answers):.1f}")
print(f"Avg context length (tokens): {sum(ctx_lengths)/len(ctx_lengths):.0f}")
print(f"Avg supporting facts       : {sum(sf_counts)/len(sf_counts):.1f}")

In [ ]:
print("\n=== 3 Example Samples ===\n")
for i, s in enumerate(experiments[:3]):
    print(f"--- Sample {i+1} ---")
    print(f"Q: {s['question']}")
    print(f"A: {s['answer']}")
    ctx_snippet = s['context'][0][1][0] if s['context'] else ''
    print(f"Context[0][0]: {ctx_snippet[:120]}...")
    print(f"Supporting facts: {s['supporting_facts']}")
    print()

In [ ]:
# Save summary and outputs to Drive (optional)
import json
import os
import shutil

summary_path = "../experiments/results/data_exploration_summary.json"
os.makedirs("../experiments/results", exist_ok=True)

if "experiments" in globals() and "calibration" in globals():
    summary = {
        "experiment_samples": len(experiments),
        "calibration_samples": len(calibration),
        "avg_question_len_words": sum(len(q.split()) for q in questions) / max(len(questions), 1),
        "avg_answer_len_words": sum(len(a.split()) for a in answers) / max(len(answers), 1),
        "avg_context_len_tokens": sum(ctx_lengths) / max(len(ctx_lengths), 1),
        "avg_supporting_facts": sum(sf_counts) / max(len(sf_counts), 1),
    }
    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)
    print(f"Saved summary to {summary_path}")
else:
    print("Run the data exploration cells first to compute summary stats.")

drive_root = "/content/drive/MyDrive/CENG467_Final"
src_dir = "../experiments/results"
dst_dir = os.path.join(drive_root, "experiments", "results")

if os.path.exists("/content/drive"):
    if os.path.exists(src_dir):
        os.makedirs(dst_dir, exist_ok=True)
        shutil.copytree(src_dir, dst_dir, dirs_exist_ok=True)
        print(f"Copied {src_dir}/ to {dst_dir}")
    else:
        print(f"Source folder not found: {src_dir}")
else:
    print("Drive not mounted. Run the Drive mount cell first.")